# Compare Airflow experiments

Enter the group ID printed by the Typer script. This notebook reads live DAG Run and XCom records
from the selected Airflow endpoints using the repository's `.env` or inherited environment
variables. It groups runs by workload so timings and result checks are compared for the same inputs.
See the [English guide](../../docs/operations-en.md) or [한국어 가이드](../../docs/operations-ko.md).


In [ ]:
import os
import re

import pandas as pd
from IPython.display import display

from pilot_kit.airflow_api import collect, connect, list_runs

group = os.environ.get("PILOT_RUN_GROUP", "")
targets = ["local", "kind", "kubernetes"]

if not re.fullmatch(r"[0-9a-f]{16}", group):
    raise ValueError("Set group to the 16-character experiment group printed by the CLI.")

In [ ]:
rows = []
coverage = []
for target in targets:
    with connect(target) as client:
        target_rows = [collect(client, target, run) for run in list_runs(client, group)]
    rows.extend(target_rows)
    coverage.append({"environment": target, "runs_found": len(target_rows)})

display(pd.DataFrame(coverage))
if not rows:
    raise ValueError("No matching runs. Check the group ID and selected environments.")

runs = pd.DataFrame(rows)
display(runs)

In [ ]:
summary = (
    runs.assign(
        succeeded=runs["state"].eq("success"),
        failed=runs["state"].eq("failed"),
        result_matches=runs["matches"].eq(True),
    )
    .groupby(["items", "shards", "environment"])
    .agg(
        observed=("run_id", "count"),
        succeeded=("succeeded", "sum"),
        failed=("failed", "sum"),
        result_matches=("result_matches", "sum"),
    )
)
display(summary)

completed = runs.loc[runs["state"].eq("success")].dropna(subset=["duration_seconds"])
if not completed.empty:
    timings = completed.groupby(["items", "shards", "environment"])["duration_seconds"].agg(
        ["median", "min", "max"]
    )
    display(timings)
    ax = (
        timings["median"]
        .unstack("environment")
        .plot.bar(ylabel="Median successful DAG duration (s)", xlabel="(items, shards)", rot=0)
    )
    display(ax.figure)